# 02 — Optimasi Inferensi: ONNX, Kuantisasi, dan Batching

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/12-deployment/02_optimasi_inferensi.ipynb)

Di produksi, setiap milidetik dan megabyte berarti biaya: server lebih banyak, tagihan cloud lebih besar, pengguna menunggu lebih lama. Notebook ini membahas tiga teknik optimasi inferensi yang paling sering dipakai, dan yang terpenting, cara **mengukurnya** dengan benar:

1. **Mengukur** latensi dengan benar (warm-up, p50/p95).
2. **ONNX** dan **ONNX Runtime**: menjalankan model tanpa PyTorch.
3. **Kuantisasi int8**: model 4× lebih kecil dan (sering) lebih cepat, dengan akurasi hampir sama.
4. **Batching** dan pemborosan padding.

Kita memakai dua model:
- Model sentimen dari notebook 01, untuk memeriksa **akurasi** sebelum dan sesudah optimasi.
- Model berukuran **MiniLM** (arsitektur BERT, 6 layer, ±23 juta parameter) dengan **bobot acak**, untuk mengukur **kecepatan dan ukuran**. Bobot acak tidak masalah: kecepatan dan ukuran tidak bergantung pada nilai bobot, dan kita tidak perlu mengunduh model.

> Hasil pengukuran sangat bergantung pada hardware (jumlah core CPU, dukungan instruksi int8). Jalankan sendiri dan percayai angkamu, bukan angka di buku/blog.

## 0. Instalasi dan File Aplikasi

In [ ]:
%pip install -q onnx onnxscript onnxruntime transformers pandas matplotlib

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
REPO_RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main/12-deployment/app"
FILE_APP = ["model.py", "latih.py", "main.py", "ui_gradio.py", "requirements.txt", "Dockerfile", ".dockerignore",
            "tests/test_api.py"]
APP = Path("app")
for f in FILE_APP:
    if not (APP / f).exists():
        (APP / f).parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{REPO_RAW}/{f}", APP / f)

sys.path.insert(0, str(APP.resolve()))                         # agar `import model`, `import main` bisa dipakai
ARTEFAK = (APP / "artefak").resolve()
os.environ["FOLDER_MODEL"] = str(ARTEFAK)                      # konfigurasi API lewat environment variable
print(f"Folder aplikasi: {APP.resolve()} | QUICK_RUN: {QUICK_RUN}")

In [ ]:
from latih import latih

if not (ARTEFAK / "model.onnx").exists():                      # latih dulu jika belum ada (±20 detik di CPU)
    print(latih(ARTEFAK, epoch=5 if QUICK_RUN else 40))
print(sorted(f.name for f in ARTEFAK.iterdir()))

## 1. Mengukur dengan Benar

Kesalahan umum saat mengukur kecepatan:
- **Tanpa warm-up**: panggilan pertama sering jauh lebih lambat (alokasi memori, kompilasi, cache). Jalankan beberapa kali dulu sebelum mengukur.
- **Hanya satu kali ukur**: waktu eksekusi bervariasi. Ulangi banyak kali dan laporkan **median (p50)** dan **p95**.
- Membandingkan pada **input berbeda** (ukuran batch, panjang teks).

In [ ]:
import time

import numpy as np
import pandas as pd

N_ULANG = 10 if QUICK_RUN else 50


def ukur(fungsi, n=N_ULANG, warmup=3):
    """Kembalikan (p50, p95) waktu eksekusi dalam milidetik."""
    for _ in range(warmup):
        fungsi()
    waktu = []
    for _ in range(n):
        t0 = time.perf_counter()
        fungsi()
        waktu.append((time.perf_counter() - t0) * 1000)
    return np.percentile(waktu, 50), np.percentile(waktu, 95)


print(f"Jumlah core CPU: {os.cpu_count()}")

## 2. ONNX dan ONNX Runtime

**ONNX** (*Open Neural Network Exchange*) adalah format standar untuk menyimpan model sebagai **graf komputasi** (daftar operasi seperti `MatMul`, `Add`, `Softmax`). Model dari PyTorch, TensorFlow, atau scikit-learn bisa diekspor ke ONNX, lalu dijalankan oleh **ONNX Runtime** di berbagai platform (server, mobile, browser) dan hardware.

Keuntungan untuk deployment:
- **Tanpa PyTorch** saat serving: image Docker kita 409 MB, sedangkan dengan PyTorch bisa beberapa GB (notebook 03).
- ONNX Runtime menerapkan **optimasi graf** (misalnya menggabungkan beberapa operasi menjadi satu).
- Alat **kuantisasi** yang matang.

Fungsi ekspor di [`app/latih.py`](app/latih.py):

In [ ]:
import inspect

import latih as modul_latih

print(inspect.getsource(modul_latih.ekspor_onnx))

`dynamic_shapes` penting: tanpa itu, model ONNX hanya menerima batch dan panjang teks yang sama persis dengan contoh saat ekspor. Kita lihat isi graf model sentimen:

In [ ]:
from collections import Counter

import onnx

graf = onnx.load(ARTEFAK / "model.onnx")
print("Input :", [(i.name, [d.dim_param or d.dim_value for d in i.type.tensor_type.shape.dim]) for i in graf.graph.input])
print("Output:", [o.name for o in graf.graph.output])
print("Operasi:", dict(Counter(n.op_type for n in graf.graph.node)))

**Selalu verifikasi** bahwa model hasil ekspor memberi output yang sama dengan model asli. Kita bandingkan pada seluruh data test NusaX (400 kalimat):

In [ ]:
from latih import NUSAX_URL, LABEL_ID
from model import Prediktor

test = pd.read_csv(f"{NUSAX_URL}/test.csv")
teks_test, label_test = test["text"].tolist(), test["label"].map(LABEL_ID).to_numpy()

pt, ox = Prediktor(ARTEFAK, backend="torch"), Prediktor(ARTEFAK, backend="onnx")
logit_pt, logit_ox = pt.logits(teks_test), ox.logits(teks_test)
print(f"Selisih maksimum logits: {np.abs(logit_pt - logit_ox).max():.2e}")
print(f"Akurasi PyTorch: {(logit_pt.argmax(1) == label_test).mean():.4f} | ONNX: {(logit_ox.argmax(1) == label_test).mean():.4f}")

## 3. Kuantisasi int8

Di modul 10 kita mengkuantisasi bobot secara manual. **Kuantisasi dinamis** ONNX Runtime melakukannya untuk seluruh model: bobot disimpan sebagai **int8** (1 byte, bukan 4 byte float32), dan perkalian matriks dihitung dengan instruksi integer yang lebih cepat di banyak CPU. "Dinamis" berarti skala aktivasi dihitung saat inference, sehingga tidak perlu data kalibrasi.

Pertama, model sentimen: bagaimana ukuran dan **akurasinya**?

In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic


def kuantisasi(sumber, tujuan):
    # DefaultTensorType: tipe cadangan jika tipe tensor perantara tidak diketahui alat kuantisasi
    quantize_dynamic(str(sumber), str(tujuan), weight_type=QuantType.QInt8,
                     extra_options={"DefaultTensorType": onnx.TensorProto.FLOAT})


FOLDER_INT8 = Path("artefak_int8")
FOLDER_INT8.mkdir(exist_ok=True)
for f in ["config.json", "vocab.json"]:
    (FOLDER_INT8 / f).write_text((ARTEFAK / f).read_text())
kuantisasi(ARTEFAK / "model.onnx", FOLDER_INT8 / "model.onnx")

ox8 = Prediktor(FOLDER_INT8, backend="onnx")
logit_8 = ox8.logits(teks_test)
satu = ["Barangnya bagus dan pengiriman cepat sekali, terima kasih!"]
baris = []
for nama, p, folder in [("PyTorch fp32", pt, ARTEFAK), ("ONNX fp32", ox, ARTEFAK), ("ONNX int8", ox8, FOLDER_INT8)]:
    p50, p95 = ukur(lambda: p.logits(satu))
    ukuran = (folder / ("model.pt" if nama.startswith("PyTorch") else "model.onnx")).stat().st_size / 1e3
    baris.append({"format": nama, "ukuran (KB)": round(ukuran), "akurasi test": (p.logits(teks_test).argmax(1) == label_test).mean(),
                  "p50 1 teks (ms)": round(p50, 3), "p95 (ms)": round(p95, 3)})
print(f"Prediksi int8 sama dengan fp32 untuk {(logit_8.argmax(1) == logit_ox.argmax(1)).mean():.1%} kalimat test")
pd.DataFrame(baris)

Model sentimen ini sangat kecil, sehingga waktu inference didominasi overhead (pembuatan tensor, pemanggilan fungsi), bukan perkalian matriks. Efek kuantisasi pada kecepatan baru terlihat pada model yang lebih besar. Kita buat model seukuran **MiniLM** dengan arsitektur BERT dari `transformers` (bobot acak, tanpa unduhan).

In [ ]:
import torch
from transformers import BertConfig, BertForSequenceClassification

N_LAYER = 2 if QUICK_RUN else 6
torch.manual_seed(0)
bert = BertForSequenceClassification(BertConfig(vocab_size=30522, hidden_size=384, num_hidden_layers=N_LAYER,
                                                num_attention_heads=12, intermediate_size=1536, num_labels=3)).eval()
print(f"{sum(p.numel() for p in bert.parameters()) / 1e6:.1f} juta parameter, {N_LAYER} layer")

# Input contoh: batch 8 kalimat × 64 token
ids = torch.randint(0, 30522, (8, 64))
mask = torch.ones_like(ids)
batch, panjang = torch.export.Dim("batch"), torch.export.Dim("panjang", max=512)
torch.onnx.export(bert, (ids, mask), "bert.onnx", input_names=["input_ids", "attention_mask"], output_names=["logits"],
                  dynamic_shapes=({0: batch, 1: panjang}, {0: batch, 1: panjang}), dynamo=True, external_data=False)
graf = onnx.load("bert.onnx")
del graf.graph.value_info[:]          # sama seperti di latih.py: agar bisa dikuantisasi
onnx.save(graf, "bert.onnx")
kuantisasi("bert.onnx", "bert_int8.onnx")
print("Ekspor dan kuantisasi selesai")

In [ ]:
import onnxruntime as ort

sesi = {"ONNX fp32": ort.InferenceSession("bert.onnx"), "ONNX int8": ort.InferenceSession("bert_int8.onnx")}
feed = {"input_ids": ids.numpy(), "attention_mask": mask.numpy()}
with torch.no_grad():
    acuan = bert(ids, attention_mask=mask).logits.numpy()

baris = []
with torch.no_grad():
    p50, p95 = ukur(lambda: bert(ids, attention_mask=mask))
ukuran_pt = sum(p.numel() * p.element_size() for p in bert.parameters()) / 1e6
baris.append({"format": "PyTorch fp32", "ukuran (MB)": round(ukuran_pt, 1), "p50 batch 8 (ms)": round(p50, 1),
              "p95 (ms)": round(p95, 1), "prediksi sama": 1.0})
for nama, s in sesi.items():
    y = s.run(None, feed)[0]
    p50, p95 = ukur(lambda: s.run(None, feed))
    berkas = "bert.onnx" if nama == "ONNX fp32" else "bert_int8.onnx"
    baris.append({"format": nama, "ukuran (MB)": round(Path(berkas).stat().st_size / 1e6, 1),
                  "p50 batch 8 (ms)": round(p50, 1), "p95 (ms)": round(p95, 1),
                  "prediksi sama": float((y.argmax(1) == acuan.argmax(1)).mean())})
hasil_bert = pd.DataFrame(baris)
hasil_bert

Yang biasanya terlihat: int8 sekitar **4× lebih kecil** (lebih murah disimpan dan dimuat, muat di perangkat kecil) dan lebih cepat di CPU. Besar percepatannya bergantung pada CPU. Prediksi tetap sama untuk hampir semua input. Untuk model sungguhan, **selalu ukur akurasi** di data test setelah kuantisasi (seperti pada model sentimen di atas), karena ada model yang lebih sensitif terhadap kuantisasi.

## 4. Batching

GPU dan CPU modern paling efisien memproses banyak data sekaligus. Kita ukur model int8 dengan berbagai ukuran batch: **latensi per batch** naik, tetapi **throughput** (kalimat per detik) juga naik, sampai titik jenuh.

In [ ]:
import matplotlib.pyplot as plt

s = sesi["ONNX int8"]
baris = []
for b in [1, 2, 4, 8, 16] if QUICK_RUN else [1, 2, 4, 8, 16, 32, 64]:
    f = {"input_ids": np.random.randint(0, 30522, (b, 64)), "attention_mask": np.ones((b, 64), dtype=np.int64)}
    p50, _ = ukur(lambda: s.run(None, f), n=max(5, N_ULANG // 2))
    baris.append({"batch": b, "latensi per batch (ms)": p50, "kalimat per detik": b / p50 * 1000})
df_batch = pd.DataFrame(baris)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(df_batch["batch"], df_batch["latensi per batch (ms)"], "o-")
ax[0].set(xscale="log", xlabel="ukuran batch", ylabel="ms", title="Latensi per batch")
ax[1].plot(df_batch["batch"], df_batch["kalimat per detik"], "o-", color="C1")
ax[1].set(xscale="log", xlabel="ukuran batch", ylabel="kalimat/detik", title="Throughput")
plt.tight_layout()
plt.show()
df_batch.round(1)

Inilah kompromi utama serving: **batch besar = throughput tinggi, tetapi setiap permintaan menunggu lebih lama**. Server produksi memakai **dynamic batching**: permintaan yang datang dalam jendela waktu singkat (misalnya 5–10 ms) dikumpulkan menjadi satu batch. Server LLM seperti vLLM melakukan versi lebih canggih (*continuous batching*).

### Pemborosan padding

Dalam satu batch, semua kalimat dipad ke panjang kalimat terpanjang. Jika panjangnya sangat bervariasi, banyak komputasi terbuang untuk token padding. Solusi sederhana: kelompokkan kalimat dengan panjang serupa (*bucketing*). Kita hitung pada kalimat NusaX:

In [ ]:
panjang = np.array([len(ox.vocab.encode(t)) for t in teks_test])


def porsi_padding(urutan, ukuran_batch=32):
    total = terpakai = 0
    for i in range(0, len(urutan), ukuran_batch):
        p = panjang[urutan[i:i + ukuran_batch]]
        total += p.max() * len(p)
        terpakai += p.sum()
    return 1 - terpakai / total


acak = np.random.default_rng(0).permutation(len(panjang))
print(f"Panjang token: min {panjang.min()}, median {int(np.median(panjang))}, maks {panjang.max()}")
print(f"Porsi token padding | batch acak: {porsi_padding(acak):.0%} | batch diurutkan panjang: {porsi_padding(np.argsort(panjang)):.0%}")

## 5. Teknik Lain

| Teknik | Ide | Kapan |
|---|---|---|
| **Model lebih kecil / distilasi** | Latih model kecil meniru model besar (misalnya DistilBERT, MiniLM) | Sering memberi percepatan terbesar |
| **GPU / akselerator** | ONNX Runtime dengan CUDA/TensorRT, atau hardware khusus | Model besar, volume tinggi |
| **Caching** | Simpan hasil untuk input yang sering berulang | Input berulang (FAQ, produk populer) |
| **Pengaturan thread** | `SessionOptions.intra_op_num_threads` sesuai jumlah core per replika | Beberapa replika dalam satu mesin |
| **Kuantisasi statis / 4-bit** | Kuantisasi aktivasi juga (dengan data kalibrasi), atau bobot 4-bit (modul 10) | Butuh kecepatan/ukuran ekstrem |
| **Server LLM khusus** | vLLM, TGI: continuous batching, KV cache (modul 08), paged attention | Menyajikan LLM sendiri |

Urutan kerja yang disarankan: **ukur dulu** di mana waktu dihabiskan (model? preprocessing? jaringan?), optimalkan bagian terbesar, lalu **ukur lagi**, termasuk akurasinya.

## Ringkasan

- Ukur dengan warm-up, ulangi banyak kali, laporkan **p50 dan p95**, pada input yang sama.
- **ONNX** membuat model portabel dan bisa dijalankan **tanpa PyTorch**; selalu verifikasi output-nya sama dengan model asli.
- **Kuantisasi int8 dinamis**: ±4× lebih kecil, sering lebih cepat di CPU; selalu cek akurasi setelahnya.
- **Batching** menaikkan throughput dengan mengorbankan latensi; kurangi pemborosan padding dengan mengelompokkan kalimat sepanjang serupa.

## Latihan

1. **Thread.** Buat `ort.SessionOptions()` dengan `intra_op_num_threads` 1, 2, dan jumlah core maksimum untuk model int8, lalu ukur latensi batch 1 dan batch 32. Kapan lebih baik memakai banyak thread per proses, dan kapan lebih baik banyak proses dengan 1 thread?
2. **Panjang sequence.** Ukur latensi model int8 untuk panjang token 16, 64, 128, 256, dan 512 (batch 8). Apakah kenaikannya linear? Hubungkan dengan kompleksitas attention (modul 08).
3. **Model sungguhan.** Ekspor model IndoBERT hasil fine-tuning di modul 06 ke ONNX (gunakan GPU Colab untuk training, lalu ekspor), kuantisasi int8, dan bandingkan ukuran, latensi, dan **macro-F1** di data test NusaX sebelum dan sesudah kuantisasi.